In [ ]:
# # TripSense — Recommendation Pipeline Testing

# ## Objective
# Test the integrated recommendation engine for tourist destinations
# in Andhra Pradesh.

# ## Test cases
# 1. New user with no rating history
# 2. Existing user with rating history
# 3. Recommendation explanations
# 4. Output structure and validation

# The pipeline combines content-based and collaborative filtering
# for personalized destination recommendations.

In [1]:
%pip install -q pandas numpy scikit-learn

In [3]:
import os

required_files = [
    "content_recommender.py",
    "collaborative_recommender.py",
    "hybrid_recommender.py",
    "explanation.py",
    "recommendation_pipeline.py",
    "tripsense_places_v2.csv",
    "tripsense_reviews_dataset.csv",
]

missing_files = [
    filename
    for filename in required_files
    if not os.path.exists(f"/content/{filename}")
]

if missing_files:
    print("Missing files:")
    for filename in missing_files:
        print("-", filename)
else:
    print("All required files are available!")

All required files are available!


In [4]:
import sys
import pandas as pd
import numpy as np
import importlib

sys.path.insert(0, "/content")
importlib.invalidate_caches()

places = pd.read_csv("/content/tripsense_places_v2.csv")
reviews = pd.read_csv("/content/tripsense_reviews_dataset.csv")

reviews["rating"] = pd.to_numeric(
    reviews["rating"],
    errors="coerce"
)

reviews = reviews.dropna(
    subset=["reviewer_name", "place_id", "rating"]
)

print("Places:", places.shape)
print("Reviews:", reviews.shape)
print("Unique users:", reviews["reviewer_name"].nunique())

Places: (444, 57)
Reviews: (2220, 11)
Unique users: 70


In [5]:
from recommendation_pipeline import RecommendationPipeline

print("RecommendationPipeline imported successfully!")

RecommendationPipeline imported successfully!


In [6]:
pipeline = RecommendationPipeline(
    places_df=places,
    interactions_df=reviews
)

print("TripSense recommendation pipeline initialized successfully!")

TripSense recommendation pipeline initialized successfully!


In [7]:
new_user_recommendations = pipeline.recommend_places(
    interests="temples, history, nature",
    user_id=None,
    top_n=10
)

display(new_user_recommendations)

,place_id,name,category,district,suitable_for,visit_duration_hrs,best_months,content_score,recommendation_method,recommendation_reason
0,AP0341,Pitikeswara Temples,temple,Markapuram,"family,spiritual,seniors",1.5,"Jan, Feb, Oct, Nov, Dec",0.235689,content-based,Pitikeswara Temples is recommended because it ...
1,AP0157,Bhairavakona,temple,Markapuram,"family,spiritual,seniors",1.5,"Jan, Feb, Nov, Dec",0.194815,content-based,Bhairavakona is recommended because it is a st...
2,AP0324,old Vishnu temples at Peddanudiyam,temple,YSR Kadapa,"family,spiritual,seniors",1.5,"Jan, Feb, Nov, Dec",0.191436,content-based,old Vishnu temples at Peddanudiyam is recommen...
3,AP0410,Maredumilli,nature,Polavaram,"adventure,nature,friends",4.0,"Jan, Feb, Nov, Dec",0.184458,content-based,Maredumilli is recommended because it matches ...
4,AP0395,Rayadurg Palace and Rama and Krishna temples,heritage,Anantapuram,"history,culture,family",1.5,"Jan, Feb, Jun, Jul, Aug, Sep, Oct, Nov, Dec",0.182596,content-based,Rayadurg Palace and Rama and Krishna temples i...
5,AP0425,Konaseema,nature,Konaseema,"adventure,nature,friends",4.0,"Jan, Feb, Nov, Dec",0.160048,content-based,Konaseema is recommended because it matches yo...
6,AP0414,Draksharamam,temple,Konaseema,"family,spiritual,seniors",1.5,"Jan, Feb, Nov, Dec",0.105970,content-based,Draksharamam is recommended because it is a st...
7,AP0222,Akkanna Madanna Caves,heritage,NTR,"history,culture,family",1.5,"Jan, Feb, Nov, Dec",0.099926,content-based,Akkanna Madanna Caves is recommended because i...
8,AP0325,Pushpagiri Group of Temples,temple,YSR Kadapa,"family,spiritual,seniors",1.5,"Jan, Feb, Nov, Dec",0.089380,content-based,Pushpagiri Group of Temples is recommended bec...
9,AP0302,"Palli Kondeswarar Temple, Surutupalle",temple,Tirupati,"family,spiritual,seniors",1.5,"Jan, Feb",0.077406,content-based,"Palli Kondeswarar Temple, Surutupalle is recom..."


In [8]:
assert not new_user_recommendations.empty, (
    "No recommendations were returned."
)

assert len(new_user_recommendations) <= 10

assert "name" in new_user_recommendations.columns
assert "recommendation_method" in new_user_recommendations.columns
assert "recommendation_reason" in new_user_recommendations.columns

assert (
    new_user_recommendations["recommendation_method"]
    == "content-based"
).all()

assert new_user_recommendations["recommendation_reason"].notna().all()

print("New-user recommendation test passed!")

New-user recommendation test passed!


In [9]:
existing_user = reviews["reviewer_name"].iloc[0]

print("Testing user:", existing_user)

existing_user_recommendations = pipeline.recommend_places(
    interests="temples, history, nature",
    user_id=existing_user,
    top_n=10
)

display(existing_user_recommendations)

print(
    "Recommendation method:",
    existing_user_recommendations["recommendation_method"].iloc[0]
)

Testing user: Santosh Kumar


,place_id,name,category,district,suitable_for,best_months,content_score,collaborative_score,hybrid_score,recommendation_method,recommendation_reason
0,AP0206,Mahatma Gandhi statue,heritage,Guntur,"history,culture,family","Jan, Feb, Nov, Dec",0.025389,1.000000,0.317772,hybrid,Mahatma Gandhi statue is recommended because i...
1,AP0198,Archaelogical Museum,heritage,Sri Potti Sriramulu Nellore,"history,culture,family","Jan, Feb, Dec",0.019989,0.942461,0.296731,hybrid,Archaelogical Museum is recommended because it...
2,AP0395,Rayadurg Palace and Rama and Krishna temples,heritage,Anantapuram,"history,culture,family","Jan, Feb, Jun, Jul, Aug, Sep, Oct, Nov, Dec",0.182596,0.520414,0.283941,hybrid,Rayadurg Palace and Rama and Krishna temples i...
3,AP0325,Pushpagiri Group of Temples,temple,YSR Kadapa,"family,spiritual,seniors","Jan, Feb, Nov, Dec",0.089380,0.705764,0.274296,hybrid,Pushpagiri Group of Temples is recommended bec...
4,AP0048,KAYAM PETA,attraction,Tirupati,"family,friends","Jan, Feb, Dec",0.000000,0.897282,0.269184,hybrid,KAYAM PETA is recommended because it offers a ...
5,AP0174,Rambilli Beach,beach,Anakapalli,"family,friends,couples","Jan, Feb, Nov, Dec",0.000000,0.896036,0.268811,hybrid,Rambilli Beach is recommended because it offer...
6,AP0003,Arch Falls,waterfall,Tirupati,"adventure,friends,nature","Jan, Feb, Dec",0.050222,0.748012,0.259559,hybrid,Arch Falls is recommended because it matches y...
7,AP0416,Mantralayam,temple,Kurnool,"family,spiritual,seniors","Jan, Feb, Aug, Oct, Nov, Dec",0.000000,0.858627,0.257588,hybrid,Mantralayam is recommended because it offers a...
8,AP0339,Ramalingeswara swamy Temple,temple,Bapatla,"family,spiritual,seniors","Jan, Feb, Nov, Dec",0.000000,0.845535,0.253660,hybrid,Ramalingeswara swamy Temple is recommended bec...
9,AP0302,"Palli Kondeswarar Temple, Surutupalle",temple,Tirupati,"family,spiritual,seniors","Jan, Feb",0.077406,0.642326,0.246882,hybrid,"Palli Kondeswarar Temple, Surutupalle is recom..."


Recommendation method: hybrid


In [10]:
assert not existing_user_recommendations.empty, (
    "No recommendations were returned."
)

assert len(existing_user_recommendations) <= 10

assert "recommendation_reason" in existing_user_recommendations.columns

assert (
    existing_user_recommendations["recommendation_method"]
    == "hybrid"
).all()

print("Existing-user recommendation test passed!")

Existing-user recommendation test passed!


In [11]:
interest_tests = [
    "waterfalls and nature",
    "temples and spirituality",
    "history, forts and heritage",
]

for interests in interest_tests:
    print(f"\nInterests: {interests}")

    results = pipeline.recommend_places(
        interests=interests,
        user_id=None,
        top_n=5
    )

    display(results[["name", "category", "district"]])


Interests: waterfalls and nature


,name,category,district
0,Talakona Waterfalls,waterfall,Tirupati
1,Sadasiva Kona waterfalls,waterfall,Tirupati
2,Maredumilli,nature,Polavaram
3,Konaseema,nature,Konaseema
4,Katiki falls,waterfall,Alluri Sitharama Raju



Interests: temples and spirituality


,name,category,district
0,Pitikeswara Temples,temple,Markapuram
1,Bhairavakona,temple,Markapuram
2,old Vishnu temples at Peddanudiyam,temple,YSR Kadapa
3,Rayadurg Palace and Rama and Krishna temples,heritage,Anantapuram
4,Mounagiri Hanuman Temple,temple,Anantapuram



Interests: history, forts and heritage


,name,category,district
0,Anupu,heritage,Palnadu
1,Garapadu,heritage,Guntur
2,Srikanth's,heritage,Guntur
3,Lingalakonda,heritage,Anakapalli
4,budda vihar,heritage,Visakhapatnam


In [12]:
print("=" * 45)
print("TRIPSENSE MEMBER 2 — TEST SUMMARY")
print("=" * 45)

print("Places loaded:", len(places))
print("Review interactions:", len(reviews))
print("Unique users:", reviews["reviewer_name"].nunique())
print("New-user test: PASSED")
print("Existing-user test: PASSED")
print("Pipeline integration: TESTED")

TRIPSENSE MEMBER 2 — TEST SUMMARY
Places loaded: 444
Review interactions: 2220
Unique users: 70
New-user test: PASSED
Existing-user test: PASSED
Pipeline integration: TESTED
